<a href="https://colab.research.google.com/github/domeGIT/ml_image_to_latex_2024/blob/main/05_analiza_gresaka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/domeGIT/ml_image_to_latex_2024

Cloning into 'ml_image_to_latex_2024'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 202 (delta 70), reused 48 (delta 17), pack-reused 61 (from 1)
Receiving objects: 100% (202/202), 73.85 MiB | 26.80 MiB/s, done.
Resolving deltas: 100% (93/93), done.


In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms.functional as TF
from torchvision import transforms

from torch.cuda.amp import autocast, GradScaler
from nltk.translate.bleu_score import corpus_bleu

import pandas as pd
import json
import os
import shutil # potrebno za google colab

from ml_image_to_latex_2024.image2latex import Text, LatexDataset, Image2LatexModel, exact_match, collate_fn, get_device, bind_gpu
from ml_image_to_latex_2024.image2latex import ConvEncoder, Decoder, Attention

In [ ]:
# raspakivanje data.tar u /content/data.tar
dst = "/content/data"

if os.path.exists(dst):
     shutil.rmtree(dst)

!cp /content/ml_image_to_latex_2024/data.tar /content/
!tar -xf /content/data.tar -C /content

Priprema log fajla za čuvanje rezultata tessta

In [ ]:
# MOUNTOVANJE DRAJVA
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# uvođenje log fajla
os.makedirs("/content/drive/My Drive/im2latex/logs", exist_ok=True)

formulas_tex_file = "/content/drive/My Drive/im2latex/logs/error_analisys.tex"
error_logger = "/content/drive/My Drive/im2latex/logs/errors_list.txt"

Uvođenje objekata i definisanje parametara potrebnih za test

In [ ]:
# konfig/parametri
BATCH_SIZE = 16
WORKERS = 4
MAX_LENGTH = 150

In [ ]:
device = get_device()

In [ ]:
# kriterijum za loss
criterion = torch.nn.CrossEntropyLoss().to(device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize(128),
    transforms.ToTensor()
])

text_processor = Text()

test_dataset = LatexDataset('/content/data/im2latex_train.csv', transform=transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    collate_fn=lambda batch: collate_fn(batch, text_processor)
)

učitajmo prethodno sačuvani model:

In [ ]:
model = torch.load("/content/ml_image_to_latex_2024/saved_models/model10.pt", map_location=device, weights_only=False)
model.eval()

Image2LatexModel(
  (encoder): ConvEncoder(
    (feature_encoder): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(128, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
    )
  )
  (decoder): Decoder(
    (embedding): Embedding(520, 80)
    (attention): Attention(
      (decoder_attention): Linear(in_features=512, out_features=512, bias=False)
      (encoder_attention): Linear(in_features=512, out_features=512, bias=False)
      (attention): Linear(in_features=512, out_features=1, bias=False)
      (softmax): Softmax(dim=-1)
    )
    (concat): Linear(in_features=592, out_features=512, bias=True)
    (rnn): LSTM(512, 512, b

In [ ]:
from pylatexenc.latexwalker import LatexWalker
from pylatexenc.latexwalker import LatexWalkerParseError

In [ ]:
def syntaxAnalysis(model, test_loader, text_processor, criterion=None, log_file=None, error_logger=None):
    model.eval()
    results = {"total":0, "correct":0}

    with torch.no_grad():
        for batch in test_loader:
            images, formulas, formula_len = bind_gpu(batch)
            batch_size = images.size(0)

            formulas_in = formulas[:, :-1]
            formulas_out = formulas[:, 1:]

            # generiši predikcije
            predicts = model.decode_greedy_batch(images, max_length=MAX_LENGTH)
            truths = [formula.tolist() for formula in formulas]

            predict_strings = [text_processor.tokenize(text_processor.int2text(p)) for p in predicts]
            truth_strings = [text_processor.tokenize(text_processor.int2text(t)) for t in truths]

            for i in range(batch_size):
              tokens_list = predict_strings[i]
              formula_string = "$$" + "".join(tokens_list) + "$$"

              truth_tokens_list = truth_strings[i]
              truth_formula_string = "$$" + "".join(truth_tokens_list) + "$$"

              results["total"] += 1
              try:
                walker = LatexWalker(formula_string)
                nodelist, pos, len = walker.get_latex_nodes()
                results["correct"] += 1
                # printamo par formula samo ako je sintaksno ispravan
                with open(formulas_tex_file, 'a') as file:
                  file.write(formula_string + "\n")
                  file.write(truth_formula_string + "\n")
                  file.write ("------------------------------\n")
              except LatexWalkerParseError as e:
                msg = str(e)
                error_type = msg.split("\n")[0]
                if error_type not in results["errors"]:
                  results["errors"][error_type] = 1
                else:
                  results["errors"][error_type] += 1
                  with open(error_logger, 'a') as file:
                    file.write(f"Prediction is invalid LaTeX: {formula_string} -> {e}" + "\n"
                    + f"Truth string: {truth_formula_string}\n")

    print(results)
    return results

In [ ]:
syntaxAnalysis(model, test_loader, text_processor, criterion, formulas_tex_file, error_logger=error_logger)

{'total': 47549, 'correct': 47549}


{'total': 47549, 'correct': 47549}

In [ ]:
def write_formulas_in_files(model, test_loader, text_processor, predicted_formulas_tex_file, truth_formulas_tex_file, criterion=None):
    model.eval()
    results = {"total":0, "correct":0}

    with torch.no_grad():
      for batch in test_loader:
        images, formulas, formula_len = bind_gpu(batch)
        batch_size = images.size(0)

        formulas_in = formulas[:, :-1]
        formulas_out = formulas[:, 1:]

        # generiši predikcije
        predicts = model.decode_greedy_batch(images, max_length=MAX_LENGTH)
        truths = [formula.tolist() for formula in formulas]

        predict_strings = [text_processor.tokenize(text_processor.int2text(p)) for p in predicts]
        truth_strings = [text_processor.tokenize(text_processor.int2text(t)) for t in truths]

        for i in range(batch_size):
          tokens_list = predict_strings[i]
          formula_string = "$$" + " ".join(tokens_list) + "$$"

          truth_tokens_list = truth_strings[i]
          truth_formula_string = "$$" + " ".join(truth_tokens_list) + "$$"

          with open(predicted_formulas_tex_file, 'a') as file:
            file.write(formula_string + "\n")
          with open(truth_formulas_tex_file, 'a') as file:
            file.write(truth_formula_string + "\n")


In [ ]:
predicted_formulas_tex_file = "/content/drive/My Drive/im2latex/logs/predicted_formulas.tex"
truth_formulas_tex_file = "/content/drive/My Drive/im2latex/logs/true_formulas.tex"

In [ ]:
write_formulas_in_files(model, test_loader, text_processor, predicted_formulas_tex_file, truth_formulas_tex_file, criterion)

In [ ]:
import subprocess
import tempfile
import os

In [ ]:
# True za ispravnu formulu, inace False
def check_latex_formula(formula):
  tex_source = f"""\\documentclass{{article}}
  \\usepackage{{amsmath, amssymb}}
  \\begin{{document}}
  {formula}
  \\end{{document}}
  """
  #with tempfile.TemporaryDirectory() as tmpdir:
  tmpdir = "/root/tmp_formulas"
  tex_path = os.path.join(tmpdir, "formula.tex")
  with open(tex_path, "w", encoding="utf-8") as f:
    f.write(tex_source)
    result = subprocess.run(
      ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", tex_path],
      cwd=tmpdir,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
    )
    return result.returncode == 0, result.stdout.decode()


In [ ]:
!apt-get install texlive-latex-base

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  dvisvgm fonts-droid-fallback fonts-lmodern fonts-noto-mono fonts-urw-base35
  libgs9 libgs9-common libidn12 libijs-0.35 libjbig2dec0 libkpathsea6
  libptexenc1 libsynctex2 libteckit0 libtexlua53 libtexluajit2 libwoff1
  libzzip-0-13 lmodern poppler-data t1utils tex-common texlive-base
  texlive-binaries xfonts-encodings xfonts-utils
Suggested packages:
  fonts-noto fonts-freefont-otf | fonts-freefont-ttf fonts-texgyre
  poppler-utils ghostscript fonts-japanese-mincho | fonts-ipafont-mincho
  fonts-japanese-gothic | fonts-ipafont-gothic fonts-arphic-ukai
  fonts-arphic-uming fonts-nanum debhelper gv | postscript-viewer perl-tk xpdf
  | pdf-viewer xzdec texlive-latex-base-doc
The following NEW packages will be installed:
  dvisvgm fonts-droid-fallback fonts-lmodern fonts-noto-mono fonts-urw-base35
  libgs9 libgs9-common libidn12 libijs-0.

In [ ]:
failed_formulas = "/content/drive/My Drive/im2latex/logs/failed_formulas.tex"
with open(predicted_formulas_tex_file, 'r') as pred_file:
  with open(truth_formulas_tex_file, 'r') as truth_file:
    for i in range(5000):
      f_predicted = pred_file.readline()
      f_true = truth_file.readline()

      ok, log = check_latex_formula(f_predicted)
      if not ok:
        with open(failed_formulas, 'a') as failed_file:
          failed_file.write(f_predicted + "\n")
          failed_file.write(f_true + "\n")
          failed_file.write("----------------------------\n")
          print(log[:400])


Streaming output truncated to the last 5000 lines.
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(/root/tmp_formulas/formula.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>)
! Emergency stop.
<*> /root/tmp_formulas/formula.tex
                                  
!  ==> Fatal error occurred, no output PDF file produc
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(/root/tmp_formulas/formula.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>)
! Emergency stop.
<*> /root/tmp_formulas/formula.tex
                                  
!  ==> Fatal error occurred, no output PDF file produc
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering exten